# ML-10 — Content Action Playbook (`Lane 1: Content Refresh`)

## 1. Ranked Actions + Reason Codes & Archetype Mapping

This playbook translates the calibrated probabilities (`decay_probability`) from our Week-5 client-grouped Random Forest model into a prioritized, human-reviewed editorial queue. Rather than exposing raw probabilities alone, every content item is mapped to an **operational archetype**, a single **reason code**, and an **action tier** calibrated against search demand (`impressions_90d`), SERP visibility (`avg_position`), and content staleness.

### Archetype → Action Mapping Matrix

| Content Archetype | Decision Criteria (Pre-Decision Signals) | Action Tier | Reason Code | Editorial Playbook Action |
| :--- | :--- | :--- | :--- | :--- |
| **High-Value At-Risk Pillar** | `decay_prob >= 0.65` AND `impressions_90d >= 1,000` AND `avg_position > 0` | **Urgent Refresh** | `RC-01: High Decay Risk + High Demand` | Priority editorial overhaul: update facts/stats, refresh title/H1 for CTR, and verify intent alignment within 7 days. |
| **Recoverable Mid-Tier Page** | `decay_prob >= 0.65` AND `100 <= impressions_90d < 1,000` AND `avg_position > 0` | **Standard Refresh** | `RC-02: Elevated Decay Risk + Visible Rank` | Scheduled quarterly update: expand thin sections, optimize internal links, and audit SERP snippet gaps. |
| **Aging Low-Traction Asset** | `decay_prob < 0.45` AND `age_days > 365` AND `impressions_90d < 100` | **Review for Retirement** | `RC-03: Stale Content + Low Search Demand` | Editorial triage: evaluate for 301 consolidation into a stronger pillar page, canonicalization, or archival. |
| **Healthy / Low-Risk Asset** | All other visible or recently updated pages | **Monitor** | `RC-04: Stable Trajectory / Low Urgency` | No immediate editorial intervention; keep in automated quarterly telemetry tracking. |

### Decay / Refresh Operational Insight
Our Week-4 and Week-5 audits measured that **neither content age (`>180d`) nor raw impression volume alone** reliably isolates recoverable decay. Conditioning high model-predicted decay probability (`>= 0.65`) on proven search visibility (`avg_position > 0` and `impressions_90d >= 100`) concentrates human editorial hours on pages where refreshing can realistically defend or recover organic clicks.

In [4]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

SEED = 42

# Localiza a raiz do repositório automaticamente
if os.path.exists("data/raw/content_refresh_anonymized.csv"):
    REPO_ROOT = "."
elif os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    REPO_ROOT = "../.."
else:
    raise FileNotFoundError(f"Dataset não encontrado a partir de {os.getcwd()}")

data_path = os.path.join(REPO_ROOT, "data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

# Define alvo observado (usado APENAS para treinar o modelo via split agrupado por cliente, nunca como feature)
if "is_declining" in df.columns:
    df["is_declining_label"] = df["is_declining"].astype(int)
elif "trend_direction" in df.columns:
    df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower().isin(["declining", "down"])).astype(int)
elif "trend_pct" in df.columns:
    df["is_declining_label"] = (df["trend_pct"] <= -20.0).astype(int)
else:
    raise KeyError(f"Coluna de alvo não identificada: {list(df.columns)}")

age_col = "days_since_last_update" if "days_since_last_update" in df.columns else "content_age_days"

# Engenharia de features estritamente pré-decisão (tratando sentinel avg_position == 0)
df["has_position_data"] = (df["avg_position"] > 0).astype(int)
pos_median = df.loc[df["avg_position"] > 0, "avg_position"].median()
df["avg_position_clean"] = df["avg_position"].replace(0, pos_median)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"].clip(lower=0))
df["clicks_per_impression"] = df["clicks_90d"] / np.maximum(df["impressions_90d"], 1)

excluded_cols = {
    "content_id", "client_id", "url", "domain", "is_declining", "is_declining_label",
    "trend_direction", "trend_pct", "avg_position"
}
feature_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in excluded_cols
]

X = df[feature_cols].fillna(0.0)
y = df["is_declining_label"].values

# Gera probabilidades fora da amostra (out-of-fold por cliente) para pontuar todas as 30.000 URLs sem vazamento
gss = GroupShuffleSplit(n_splits=5, test_size=0.20, random_state=SEED)
oof_probs = np.zeros(len(df), dtype=float)
oof_counts = np.zeros(len(df), dtype=int)

for tr_idx, te_idx in gss.split(X, y, groups=df["client_id"]):
    rf = RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
    rf.fit(X.iloc[tr_idx], y[tr_idx])
    oof_probs[te_idx] += rf.predict_proba(X.iloc[te_idx])[:, 1]
    oof_counts[te_idx] += 1

# Fallback para qualquer cliente que não caiu nos 5 folds de teste
unscored_mask = oof_counts == 0
if unscored_mask.any():
    rf_full = RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
    rf_full.fit(X.iloc[~unscored_mask], y[~unscored_mask])
    oof_probs[unscored_mask] = rf_full.predict_proba(X.iloc[unscored_mask])[:, 1]
    oof_counts[unscored_mask] = 1

df["decay_probability"] = np.round(oof_probs / oof_counts, 4)

# Mapeamento Archetype -> Action + Reason Code
def map_playbook_action(row):
    prob = row["decay_probability"]
    impr = row["impressions_90d"]
    age = row[age_col]
    has_pos = row["avg_position"] > 0

    if prob >= 0.65 and impr >= 1000 and has_pos:
        return "High-Value At-Risk Pillar", "Urgent Refresh", "RC-01: High Decay Risk + High Demand"
    elif prob >= 0.65 and impr >= 100 and has_pos:
        return "Recoverable Mid-Tier Page", "Standard Refresh", "RC-02: Elevated Decay Risk + Visible Rank"
    elif prob < 0.45 and age > 365 and impr < 100:
        return "Aging Low-Traction Asset", "Review for Retirement", "RC-03: Stale Content + Low Search Demand"
    else:
        return "Healthy / Low-Risk Asset", "Monitor", "RC-04: Stable Trajectory / Low Urgency"

mapped = df.apply(map_playbook_action, axis=1, result_type="expand")
df[["archetype", "recommended_action", "reason_code"]] = mapped

# Ordena a fila por prioridade de ação e impacto esperado (probabilidade * log_impressions)
action_priority = {
    "Urgent Refresh": 1,
    "Standard Refresh": 2,
    "Review for Retirement": 3,
    "Monitor": 4,
}
df["action_priority"] = df["recommended_action"].map(action_priority)
df["expected_impact_score"] = np.round(df["decay_probability"] * df["log_impressions_90d"], 4)

ranked_queue = df.sort_values(
    by=["action_priority", "expected_impact_score", "decay_probability"],
    ascending=[True, False, False],
).reset_index(drop=True)
ranked_queue["queue_rank"] = ranked_queue.index + 1

print("=== PLAYBOOK ACTION DISTRIBUTION (ALL 30,000 CONTENT ITEMS) ===")
dist_summary = (
    ranked_queue.groupby(["action_priority", "recommended_action", "reason_code"])
    .agg(
        url_count=("content_id", "count"),
        pct_of_corpus=("content_id", lambda s: round(100.0 * len(s) / len(ranked_queue), 2)),
        median_impressions_90d=("impressions_90d", "median"),
        mean_decay_prob=("decay_probability", lambda s: round(float(s.mean()), 4)),
    )
    .reset_index()
    .drop(columns=["action_priority"])
)
print(dist_summary.to_string(index=False))

print("\n=== TOP 10 PRIORITIZED QUEUE ITEMS ===")
preview_cols = [
    "queue_rank", "content_id", "client_id", "recommended_action",
    "reason_code", "decay_probability", "impressions_90d", "avg_position", age_col
]
print(ranked_queue[preview_cols].head(10).to_string(index=False))

=== PLAYBOOK ACTION DISTRIBUTION (ALL 30,000 CONTENT ITEMS) ===
   recommended_action                               reason_code  url_count  pct_of_corpus  median_impressions_90d  mean_decay_prob
       Urgent Refresh      RC-01: High Decay Risk + High Demand       3982          13.27                  2488.5           0.6930
     Standard Refresh RC-02: Elevated Decay Risk + Visible Rank       4209          14.03                   371.0           0.7078
Review for Retirement  RC-03: Stale Content + Low Search Demand          2           0.01                     1.0           0.0975
              Monitor    RC-04: Stable Trajectory / Low Urgency      21807          72.69                   569.0           0.4769

=== TOP 10 PRIORITIZED QUEUE ITEMS ===
 queue_rank           content_id         client_id recommended_action                          reason_code  decay_probability  impressions_90d  avg_position  days_since_last_update
          1 content_453722754fea client_f369cb89fc     Urgen

## 2. Intended Use, Cost/Value Thinking, and Limits

* **Intended Use (Decision-Support Only):** Designed for weekly SEO and content-operations triage across multi-client portfolios. The queue ranks candidates so human editors allocate limited writing capacity to high-impression, visible pages showing elevated decay risk.
* **Cost / Value Asymmetric Trade-off:**
  * *False Positive Cost (Low–Moderate):* An editor spends ~1–2 hours reviewing a stable evergreen page flagged as `Urgent Refresh`, verifies its accuracy, and leaves it untouched (`Monitor`).
  * *False Negative Cost (High):* Failing to catch decay on a top-ranking pillar page (`impressions_90d >= 1,000`) results in sustained organic click loss that takes months of re-indexing to recover. Therefore, our playbook favors higher recall on high-impression pillars (`Urgent Refresh`) while gating destructive actions behind human review.
* **Known Limits & Assumptions:**
  * *Associational, Not Causal:* A high `decay_probability` indicates historical patterns associated with traffic drops; it does not guarantee that rewriting copy will reverse external SERP layout changes or seasonal demand shifts.
  * *Static Snapshot Horizon:* Features summarize a 90-day trailing window; sudden intra-week algorithm updates or competitor launches require daily warehouse monitoring (`fact_content_daily_performance`).

---

## 3. Human Review + The No-Go List

### Mandatory Human-in-the-Loop Verification Rules
Before executing any refresh or retirement recommendation, an editor must verify:
1. **SERP & Intent Check:** Confirm whether CTR/impression pressure comes from outdated content versus non-actionable SERP features (Featured Snippets, AI Overviews, local packs).
2. **Cannibalization Check:** Verify that another URL on the same client domain is not absorbing impressions for the primary query cluster.
3. **Conversion / Business Value Check:** Cross-check GA4 conversion value before retiring low-impression pages (`RC-03`), as niche bottom-of-funnel pages often convert despite low search volume.

### The No-Go List (What Must NEVER Be Automated)
1. **Automated Unpublishing, Deletion, or 301 Redirects:** `Review for Retirement` (`RC-03`) is strictly an editorial flag; automated deletion risks breaking backlink equity and conversion funnels.
2. **Autonomous LLM Rewriting & Publishing:** Model scores must never trigger unreviewed AI text generation directly to production CMS pages.
3. **Legal, Compliance, Medical/Financial (YMYL), and Brand-Canonical Pages:** Regulated or contractual pages are excluded from automated content modification regardless of decay score.
4. **Unranked / Zero-Telemetry Pages (`avg_position == 0`):** Pages lacking valid SERP ranking telemetry must never be pushed into `Urgent Refresh`.

---

## 4. Monitoring / Retrain Triggers

To prevent silent model staleness in production workflows, track four lightweight operational triggers:
1. **Queue Volume / Prediction Drift:** Trigger a threshold audit if `Urgent Refresh` exceeds **15%** or drops below **1%** of eligible visible URLs in any weekly scoring run.
2. **Feature Distribution Shift:** Alert if the share of rows with `avg_position == 0` or zero-CTR shifts by more than **±10 percentage points** relative to the training baseline (indicating GSC/GA4 ingestion changes).
3. **Editorial Acceptance Rate (Human Feedback Loop):** Track the percentage of `Urgent Refresh` recommendations accepted by editors during weekly triage; investigate if acceptance falls below **50%**.
4. **Quarterly Backtest Retrain Trigger:** Re-run the client-grouped validation pipeline (`w05_model.ipynb` / `w06_validation_audit.ipynb`) every quarter—or after major Google core algorithm updates—and retrain if held-out `PR-AUC` degrades by more than **0.05** (`5 percentage points`) relative to the recorded benchmark.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import matplotlib.pyplot as plt

outputs_dir = os.path.join(REPO_ROOT, "work/outputs")
figures_dir = os.path.join(REPO_ROOT, "work/figures")
os.makedirs(outputs_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

# 1. Exporta a fila priorizada (fica fora do git por regra do CI leak-guard)
export_cols = [
    "queue_rank",
    "content_id",
    "client_id",
    "archetype",
    "recommended_action",
    "reason_code",
    "decay_probability",
    "expected_impact_score",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    age_col,
]
csv_export_path = os.path.join(outputs_dir, "w07_action_playbook_queue.csv")
ranked_queue[export_cols].to_csv(csv_export_path, index=False)

# 2. Gera figura limpa e public-safe para reutilizar no paper da Semana 8
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

action_order = ["Urgent Refresh", "Standard Refresh", "Review for Retirement", "Monitor"]
counts = [int((ranked_queue["recommended_action"] == a).sum()) for a in action_order]
colors = ["#d9534f", "#f0ad4e", "#5bc0de", "#7f8c8d"]

bars = axes[0].bar(action_order, counts, color=colors, edgecolor="black", alpha=0.88)
axes[0].set_title("Playbook Queue Distribution by Action Tier (n = 30,000)", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Number of Content Items")
axes[0].tick_params(axis="x", rotation=18)
for bar, c in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 250, f"{c:,}", ha="center", va="bottom", fontsize=9)

# Segundo gráfico: mediana de impressões 90d por ação
med_impr = [float(ranked_queue.loc[ranked_queue["recommended_action"] == a, "impressions_90d"].median()) for a in action_order]
bars2 = axes[1].bar(action_order, med_impr, color=colors, edgecolor="black", alpha=0.88)
axes[1].set_title("Median 90-Day Impressions by Action Tier", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Median Impressions (90d)")
axes[1].tick_params(axis="x", rotation=18)
for bar, m in zip(bars2, med_impr):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 15, f"{m:,.0f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
fig_path = os.path.join(figures_dir, "w07_playbook_action_distribution.png")
plt.savefig(fig_path, dpi=150)
plt.close(fig)

# 3. Exporta recibo JSON com os números exatos do playbook para o paper da Semana 8
playbook_receipt = {
    "notebook": "w07_action_playbook.ipynb",
    "total_urls_scored": int(len(ranked_queue)),
    "distinct_clients": int(ranked_queue["client_id"].nunique()),
    "action_breakdown": dist_summary.to_dict(orient="records"),
    "exported_figure": "work/figures/w07_playbook_action_distribution.png",
}
json_export_path = os.path.join(outputs_dir, "w07_playbook_metrics.json")
with open(json_export_path, "w", encoding="utf-8") as f:
    json.dump(playbook_receipt, f, indent=2)

assert os.path.exists(csv_export_path), "Erro: CSV da fila não foi gerado."
assert os.path.exists(fig_path), "Erro: figura PNG não foi gerada."
assert os.path.exists(json_export_path), "Erro: recibo JSON não foi gerado."
assert "is_declining_label" not in ranked_queue[export_cols].columns, "Erro: target vazado no CSV exportado."

print(f"✓ Fila exportada (fora do git): {csv_export_path} ({len(ranked_queue):,} linhas)")
print(f"✓ Figura salva para o paper:    {fig_path}")
print(f"✓ Recibo JSON salvo para git:   {json_export_path}")

✓ Fila exportada (fora do git): ../../work/outputs/w07_action_playbook_queue.csv (30,000 linhas)
✓ Figura salva para o paper:    ../../work/figures/w07_playbook_action_distribution.png
✓ Recibo JSON salvo para git:   ../../work/outputs/w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.